# 05 · 계산기와 검색을 붙인 단일 루프
**4세션 · 도구 결과도 컨텍스트다** — 실습 40분

모델은 *이 도구를 이 값으로 불러 달라*고 말할 뿐이고, **실행은 우리 코드가 합니다.**
매 바퀴 `tool_result`가 messages에 쌓입니다 — 그것이 다음 바퀴의 컨텍스트입니다.

에이전트 = **모델 + 도구 + 루프**

In [ ]:
import os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")          # day 폴더로 이동 — data/ · prompts/ · src/ 가 여기 있다

from common import llm      # 공용 API 도구 (ETRI_LLM_3days/common/llm.py)
print("모델:", llm.MODEL, "| 모의 모드" if llm.MOCK else "| 실제 호출")
from src import rag
print("임베딩:", "간이 해시 (EMBED=hash)" if rag.EMBED == "hash" else rag.ST_MODEL)
import json, inspect
from src import loop

loop.SEARCH_INDEX = rag.Index.build(rag.chunk_folder("data/kb"))

## 1. 함수 정의의 실물
이름 · 설명 · 입력 스키마 세 가지가 전부입니다. **설명은 모델이 도구를 고르는 유일한 근거**입니다.

In [ ]:
print(json.dumps(loop.TOOLS, ensure_ascii=False, indent=2))

> 문자열 수식을 받아 `eval`로 계산하는 도구는 만들지 않습니다. 모델이 무엇을 넘길지 우리가 통제할 수 없기 때문입니다. `enum`으로 고를 수 있는 값을 제한합니다.

## 2. 루프 코드

In [ ]:
print(inspect.getsource(loop.run))

## 3. 1단계 · 붙이기
질문 세 개로 돌려 **매 바퀴 출력**을 봅니다. 셋째 질문 — 두 도구를 차례로 부르는가?

In [ ]:
Q1 = "AX-2041의 무게는 얼마인가요?"
Q2 = "1,234,500원을 3명이 똑같이 나누면 1명당 얼마인가요?"
Q3 = "서울로 3박 출장을 가면 숙박비 한도는 모두 얼마인가요?"

for q in [Q1, Q2, Q3]:
    print("=" * 70, "\nQ", q)
    answer, trace = loop.run(q)
    print("→ 쓴 도구:", loop.tools_used(trace), "\n")

## 4. 2단계 · 일부러 틀리게
도구 설명을 *"검색한다"* / *"계산한다"*로 줄입니다. 코드는 그대로입니다.
모델이 **도구를 안 부르거나, 잘못된 도구를 고르는** 질문을 찾습니다.

In [ ]:
print(json.dumps([{t["name"]: t["description"]} for t in loop.TOOLS_VAGUE], ensure_ascii=False))

TRY = [
    Q3,
    "법인카드 사용 한도는 얼마인가요?",              # 자료에 없는 질문 — 지어내는가
    "자기계발비 한도를 12개월로 나누면 한 달에 얼마인가요?",
    "회의실은 하루에 몇 번 예약할 수 있나요?",
]
for q in TRY:
    print("=" * 70, "\nQ", q)
    _, trace = loop.run(q, tools=loop.TOOLS_VAGUE, system=None)
    print("→ 쓴 도구:", loop.tools_used(trace), "\n")

## 5. 설명 문장만 고쳐 바로잡는다
잘못 고른 질문을 하나 정하고, **description만** 고쳐 다시 돌립니다. 코드는 건드리지 않습니다.

In [ ]:
MY_TOOLS = [
    {**loop.TOOLS_VAGUE[0], "description": "계산한다."},      # ✏️ 이 문장만 고친다
    {**loop.TOOLS_VAGUE[1], "description": "검색한다."},      # ✏️ 이 문장만 고친다
]
Q = TRY[2]                                                    # ✏️ 잘못 골랐던 질문

for name, tools in [("고치기 전", loop.TOOLS_VAGUE), ("고친 뒤", MY_TOOLS)]:
    print("=" * 70, f"\n[{name}]")
    _, trace = loop.run(Q, tools=tools, system=None)
    print("→ 쓴 도구:", loop.tools_used(trace))

| 질문 | 고치기 전 고른 도구 | 고친 설명 문장 | 고친 뒤 고른 도구 |
|---|---|---|---|
| | | | |

## 6. 창에 무엇이 쌓였나
마지막 실행의 messages를 열어 봅니다. 도구 결과가 그대로 창에 들어가 있습니다 — 그래서 **도구 결과도 컨텍스트**입니다.

In [ ]:
# loop.run 안의 msgs를 보려면 한 바퀴씩 직접 돌려 본다
msgs = [{"role": "user", "content": Q3}]
r = loop._create(model=llm.MODEL, max_tokens=800, tools=loop.TOOLS, messages=msgs, system=loop.SYSTEM)
for b in r.content:
    print(b.type, getattr(b, "name", ""), getattr(b, "input", getattr(b, "text", ""))) 

## 내 loop.py에 없는 것 → 내일

| 전략 | 오늘의 loop.py | Claude Code |
|---|---|---|
| 고르기 | 내가 짠 검색 하나 | 파일 검색 도구 · 스킬은 쓸 때만 본문을 불러온다 |
| 줄이기 | 없음 — messages가 끝없이 쌓인다 | 오래된 도구 출력부터 비우고, 필요하면 대화를 요약한다 |
| 쓰기 | 없음 — 끝나면 사라진다 | CLAUDE.md · 자동 메모리를 세션마다 읽어 온다 |
| 나누기 | 없음 — 창이 하나뿐 | 서브에이전트가 자기 창에서 일하고 요약만 돌려준다 |

내일은 오른쪽 열을 다 갖춘 루프 위에서 시작합니다.